# Combined hypothesis testing

This notebook combines and integrates the entire analysis workflow. It aims to test whether there was any statistically significant
- change in potential reproducibility of AGILE papers after the introduction of the guidelines in 2020 (reseach question 1)
- change in potential reproducibility of GIScience papers after that introduction (research question 2)
- difference in change between the two conference series (research question 3)

This means that we have to consider four groups of papers: AGILE and GIScience, each with a pre-guidelines group, and a post-guidelines group. 

To determine the statistical method to test these hypotheses, we first have to determine whether these groups are independent, because it is likely that authors appear in more than one of the groups. See also the authors analysis in `07_authorship.ipynb`.

# Preliminaries: preparing data, checking group independence, defining main analysis function

This sections loads and prepares the input data and checks the independence of the groups. All later sections in this notebook depend on this one, but are independent from one another. That means you have to run the code in this section only once and can later explore different variable settings for the hypotheses without interfering with each other. 

## Setup and prepare data

You can set global variables after the necessary libraries are loaded, then the data is loaded and preprocessed. 

The script assumes that the available data is in two CSV files that use the same column name and ID for the papers. For short explanation of other relevant columns/attributes, see code below. 

The preprocessing requires the same two steps for all hypotheses:

1. We need to remove all conceptual papers and papers from the transitional years.
2. For basic calculations on ranks (see discussion on that in a later section), we need to convert the UDAO scheme into ranks 0 to 3.

In [7]:
# all Python imports
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.contingency_tables import Table2x2
from IPython.display import display
import os

In [ ]:
#pd.set_option('future.no_silent_downcasting', True)
pd.options.display.float_format = "{:.3f}".format

# change variables as needed
PAPER_CSV = 'data-clean/all-data.csv'
PID_COL = 'paper' # column that holds the paper id in both files; values need to be identical for joining
CONF_COL = 'conf' # column that contains the two conferences (or main groups to be compared)
GROUP_A = 'agile' # the value of CONF_COL to use for first main group
GROUP_B = 'giscience' # the value of CONF_COL to use for second main group
YEAR_COL = 'year' # column that contains the year of the conference (for creating secondary groups pre-/post-intervention)
GROUP_A1 = [2017, 2018, 2019] # years to use for creating secondary group pre-intervention from first main group
GROUP_A2 = [2021, 2022, 2023, 2024] # years to use for creating secondary group post-intervention from first main group
GROUP_B1 = [2016, 2018] # years to use for creating secondary group pre-intervention from second main group
GROUP_B2 = [2021, 2023] # years to use for creating secondary group post-intervention from second main group
FILTER_COL = 'consolidated_cp' # column name used for filtering, e.g., conceptual papers; BOOLEAN mask
TEST_COL = ['consolidated_data', 'consolidated_methods', 'consolidated_results'] # criteria columns to be read and converted
CRITERIA = ['data', 'methods', 'results'] # for creating new columns with converted values of TEST_COL, has to be same length as TEST_COL
REPLACE = {
    'Not applicable': np.nan,
    'U': 0,
    'D': 1,
    'A': 2,
    'O': 3
}

AUTHOR_CSV = 'data-clean/authors.csv'
AID_COL = 'identity'

P_VALUE=0.01

### Author group mapping

Each paper in `authors.csv` is assigned a coarse group label (A–D, or X for excluded years) that reflects which analytical group it belongs to.
This label is used **exclusively** to trace author overlap across groups — it does not enter the statistical models.
The models are built from `all-data.csv` using the year-based group variables defined above (`GROUP_A1`, `GROUP_A2`, `GROUP_B1`, `GROUP_B2`).

| Group | Conference | Years | Analytical role |
|-------|------------|-------|-----------------|
| A | AGILE | 2017, 2018, 2019 | Pre-intervention AGILE (`group_a1`) |
| B | AGILE | 2021, 2022, 2023, 2024 | Post-intervention AGILE (`group_a2`) |
| C | GIScience | 2016, 2018 | Pre-intervention GIScience (`group_b1`) |
| D | GIScience | 2021, 2023 | Post-intervention GIScience (`group_b2`) |
| X | both | 2020 | Transitional year — excluded from all analysis |

The mapping is derived programmatically from the configuration variables above, so it stays in sync if year assignments are changed.
If `authors.csv` does not yet have a `group` column, the cell below adds it.

In [9]:
def assign_author_group(conf, year):
    """Return the coarse author group label for a paper given its conference and year.

    Groups A-D map to the four analytical sub-groups defined by the configuration
    variables above. X marks the transitional year (2020) that is excluded from all
    statistical analysis. The group label is used only to check author overlap across
    groups; it is not a model input.
    """
    if year in GROUP_A1 and conf == GROUP_A:
        return 'A'
    if year in GROUP_A2 and conf == GROUP_A:
        return 'B'
    if year in GROUP_B1 and conf == GROUP_B:
        return 'C'
    if year in GROUP_B2 and conf == GROUP_B:
        return 'D'
    return 'X'

# Add 'group' column to authors.csv if not already present.
# authors.csv contains conf and year columns, so the group can be derived directly.
_df_authors_raw = pd.read_csv(AUTHOR_CSV)
if 'group' not in _df_authors_raw.columns:
    _df_authors_raw['group'] = _df_authors_raw.apply(
        lambda r: assign_author_group(r[CONF_COL], r[YEAR_COL]), axis=1
    )
    _df_authors_raw.to_csv(AUTHOR_CSV, index=False)
    print(f"'group' column added to {AUTHOR_CSV}")
    print(_df_authors_raw['group'].value_counts().sort_index())
else:
    print(f"'group' column already present in {AUTHOR_CSV}, skipping")

'group' column already present in data-clean/authors.csv, skipping


In [10]:
# first the data on the papers

# Load the CSV file
df_papers = pd.read_csv(PAPER_CSV)

# remove conceptual papers
mask = df_papers[FILTER_COL] == True
filtered_df = df_papers[~mask].copy()

# convert values to numbers
for i in range(len(TEST_COL)):
    filtered_df.loc[:, CRITERIA[i]] = filtered_df.loc[:, TEST_COL[i]].replace(REPLACE).astype('Int64')

# Create the four groups
df_A = filtered_df[filtered_df[CONF_COL] == GROUP_A]
group_a1 = df_A[df_A[YEAR_COL].isin(GROUP_A1)][CRITERIA + [PID_COL]].dropna()
group_a1["group"] = "A1"
group_a2 = df_A[df_A[YEAR_COL].isin(GROUP_A2)][CRITERIA + [PID_COL]].dropna()
group_a2["group"] = "A2"
print('Groups ' + GROUP_A +
      ': n = ' + str(len(group_a1)) + ' (Pre-Intervention) & n = ' + str(len(group_a2)) + ' (Post-Intervention)')

df_B = filtered_df[filtered_df[CONF_COL] == GROUP_B]
group_b1 = df_B[df_B[YEAR_COL].isin(GROUP_B1)][CRITERIA + [PID_COL]].dropna()
group_b1["group"] = "B1"
group_b2 = df_B[df_B[YEAR_COL].isin(GROUP_B2)][CRITERIA + [PID_COL]].dropna()
group_b2["group"] = "B2"
print('Groups ' + GROUP_B + 
      ': n = ' + str(len(group_b1)) + ' (Pre-Intervention) & n = ' + str(len(group_b2)) + ' (Post-Intervention)')

group_a1.head()

NameError: name 'PAPER_CSV' is not defined

In [ ]:
# then the data on the authors

# load the CSV file
df_authors = pd.read_csv(AUTHOR_CSV)

# filter for relevant columns
df_authors = df_authors[[PID_COL, AID_COL]]

df_authors.head()

## Check authors' cross-publications

The `group` column in `authors.csv` (A/B/C/D/X, as defined in the mapping table above) is used here solely to determine which analytical group each author's papers belong to, so we can count how many papers have at least one author who also published in a different group.
This is a diagnostic step to assess whether the four analytical groups are statistically independent — it does not feed into the statistical models in later sections, which use the year-based group assignments from `all-data.csv`.

In [ ]:
# first (re-)create the main groups
check_A = pd.concat([group_a1,group_a2], ignore_index=True)
check_B = pd.concat([group_b1,group_b2], ignore_index=True)


# then add the groups where each author published
auth_grp_A = (df_authors.merge(check_A, on=PID_COL, how='inner')  # join papers to get group per author
            .groupby(AID_COL)['group']
            .agg(lambda x: set(x))
            .reset_index()
            .rename(columns={'group': 'groups'}))

auth_grp_B = (df_authors.merge(check_B, on=PID_COL, how='inner')  # join papers to get group per author
            .groupby(AID_COL)['group']
            .agg(lambda x: set(x))
            .reset_index()
            .rename(columns={'group': 'groups'}))

# then combine
merged_A = (df_authors.merge(check_A, on=PID_COL, how='inner').merge(auth_grp_A, on=AID_COL, how='inner'))
merged_B = (df_authors.merge(check_B, on=PID_COL, how='inner').merge(auth_grp_B, on=AID_COL, how='inner'))

merged_A

In [ ]:
# finally analyze and present
merged_A['other'] = merged_A.apply(lambda row: row['groups'] - {row['group']}, axis=1)
paper_other_A = (merged_A.groupby([PID_COL, 'group'])['other']
                     .agg(lambda sets: set().union(*sets))
                     .reset_index()
                     .rename(columns={'group': 'paper_group', 'other': 'other_groups'}))
paper_other_A['has_cross_group_author'] = paper_other_A['other_groups'].apply(bool)

merged_B['other'] = merged_B.apply(lambda row: row['groups'] - {row['group']}, axis=1)
paper_other_B = (merged_B.groupby([PID_COL, 'group'])['other']
                     .agg(lambda sets: set().union(*sets))
                     .reset_index()
                     .rename(columns={'group': 'paper_group', 'other': 'other_groups'}))
paper_other_B['has_cross_group_author'] = paper_other_B['other_groups'].apply(bool)
                                                                              
pd.crosstab(paper_other_A["has_cross_group_author"], paper_other_A["paper_group"])

In [ ]:
pd.crosstab(paper_other_B["has_cross_group_author"], paper_other_B["paper_group"])

The numbers match those from initial dataframes (A1 = AGILE pre, A2 = AGILE post, etc.) so we can assume that there are no semantic errors in the logic. The numbers show that between one third and more than half of the papers have at least one (co-) author who also published in the other group, violating the assumption of independence, increasing correlation and possibly inflating effect sizes when ignored. 

Let's look at cross-conference publishing next. 

In [ ]:
# first create one main group again
check = pd.concat([check_A,check_B], ignore_index=True)

# then add the groups where the author published
auth_grp = (df_authors.merge(check, on=PID_COL, how='inner')  # join papers to get group per author
            .groupby(AID_COL)['group']
            .agg(lambda x: set(x))
            .reset_index()
            .rename(columns={'group': 'groups'}))

# Merge back to original data
merged = (df_authors.merge(check, on=PID_COL, how='inner')
          .merge(auth_grp, on=AID_COL, how='inner'))

# Compute 'other_groups' per author-paper
merged['other'] = merged.apply(lambda r: r['groups'] - {r['group']}, axis=1)

# Aggregate per paper: union of all authors' 'other' sets
paper_other = (merged.groupby([PID_COL,'group'])['other']
                   .agg(lambda sets: set().union(*sets))
                   .reset_index()
                   .rename(columns={'group':'paper_group','other':'other_groups'}))
paper_other['has_cross_group_author'] = paper_other['other_groups'].apply(bool)

pd.crosstab(paper_other["has_cross_group_author"], paper_other["paper_group"])

The difference with numbers above indicated cross-conference publications. For example, for group A1, three more papers (24 vs. 21) have at least one (co-)author in another group, which has to be B1 or B2 (GIScience). For A2, it's a larger increase of 8, and for B1 and B2 it's 7 and 6, respectively. A deeper analysis is beyond the scope of this study, but the overall conclusions are:

1. Too many papers have authors who have published before and after the intervention to assume independence of the groups. 
2. Many papers of GIScience in particular have authors who also published at AGILE, increasing the possibility of a spill-over effect of the AGILE guidelines into GIScience practices. 

## Main analysis function

To recap, the data for testing our research questions consists of:

- the unit of analysis are papers
- for each paper, we consider its group, its authors, and three ordinal (ranked) criteria (data, methods, results)
- a higher numerical value for a rank is better (i.e., 3 is better than 2), but distance between ranks is not uniform, and a rank of 2 is not "twice as good" as a rank of 1)
- each paper belongs to one of four groups: we have a pre- and a post-intervention period for each of the two conferences
- all groups have different sizes, the smallest has n = 24
- each paper has one or more authors
- several papers have at least one (co-)author who contributed to one or more papers in other groups, thus the groups are not completely independent

To address research questions 1 and 2, we compare the pre- and post-intervention groups within a conference. Since the authors are shared across groups and the unit of interest are the papers, we model the authors as random effects using an ordinal mixed effects model: A cumulative link model that represents for each paper's criteria the odds of being in a higher rank, cumulated across all rank thresholds, influenced by all its authors' random effects.

To address research question 3, we compare the effect sizes of the above analysis, i.e., the odds ratios of having a higher rank for a criterion. However, a cumulative link model assumes that the effect of the predictor is the same across all thresholds between ranks. In order to be able to correctly interpret the odds ratios, we also need to check whether this proportional odds assumption holds. 

Unfortunately, to our knowledge, a cumulative linked mixed model is not available in a Python libray yet. Because of contributor preference and skills, we decided to use rpy2. This reduces ease of reproducibility of this notebook, because it requires a more complicated computational environment including a Python and R installation.

The following function takes as input two Pandas dataframes (one for pre-processed papers, one for authors) and returns all the required information:

- a fitted model
- Likelihood ratio test for group effects
- estimated marginal means for a post-hoc comparison of each criterion
- the odds ratios for comparing effect sizes between conferences (our third hypothesis), adjusted for author random effects.
- a check whether the proportional odds assumption holds.

In [ ]:
def analyze_group(group, authors):
    # Merge to long format (one row per paper-author-criterion)
    long = group.melt(
        id_vars=[PID_COL, "group"],
        value_vars=CRITERIA,
        var_name="criterion",
        value_name="rank"
    )
    
    # Merge authors: each paper-criterion row is replicated per author
    long = long.merge(authors, on=PID_COL)

    # Ensure correct types
    long["rank"] = pd.Categorical(long["rank"].astype(str),
                                   categories=["0", "1", "2", "3"],
                                   ordered=True)
    long["group"] = long["group"].astype(str)
    long["criterion"] = long["criterion"].astype(str)
    long[AID_COL] = long[AID_COL].astype(str)
    long[PID_COL] = long[PID_COL].astype(str)

    # Convert column by column instead of the whole dataframe
    with localconverter(default_converter):
        r_df = robjects.r['data.frame'](
            **{col: robjects.StrVector(long[col].tolist())
               for col in long.columns}
        )
    robjects.globalenv["dat"] = r_df

    # Convert to ordered factor in R
    robjects.r('''
        dat$rank      <- factor(dat$rank, levels=c("0","1","2","3"), ordered=TRUE)
        dat$group     <- factor(dat$group)
        dat$criterion <- factor(dat$criterion)
        dat$identity <- factor(dat$identity)
        dat$paper  <- factor(dat$paper)
    ''')

    # Initial model fit:
    #   Fixed effects:  group * criterion (interaction: does group effect differ per criterion?)
    #   Random effects: (1|author_id) + (1|paper_id)
    #     - author_id: accounts for authors publishing in both groups
    #     - paper_id:  accounts for multiple authors per paper sharing the same outcome
    #   However, did not converge.
    # Revised model fit: drop paper_id random effect (author_id is the more important one)
    robjects.r('''
        model <- clmm(
            rank ~ group * criterion + (1 | identity),
            data = dat,
            link = "logit",
            Hess = TRUE
        )    
        model_null <- clmm(
            rank ~ criterion + (1 | identity),
            data   = dat,
            link   = "logit",
            Hess   = TRUE
        )
    ''')

    # Extract results
    print("=== Model Summary ===")
    print(robjects.r("capture.output(summary(model))"))

    print("\n=== Likelihood Ratio Test: group effect ===")
    robjects.r('''
        ll_full <- as.numeric(logLik(model))
        ll_null <- as.numeric(logLik(model_null))
        df_diff <- attr(logLik(model), "df") - attr(logLik(model_null), "df")
        chi2    <- -2 * (ll_null - ll_full)
        p_val   <- pchisq(chi2, df = df_diff, lower.tail = FALSE)
        cat("Chi2:", round(chi2, 3), " df:", df_diff, " p:", round(p_val, 4), "\n")
    ''')

    print("\n=== Pairwise group comparisons per criterion ===")
    robjects.r('''
        library(emmeans)
        emm <- emmeans(model, pairwise ~ group | criterion, mode="linear.predictor")
        print(summary(emm$contrasts))
    ''')

    # Effect sizes: Odds ratios with 95% CI from model coefficients (model-based effect size)
    print("\n=== Odds Ratios (model-based effect size) ===")
    robjects.r('''
        coefs <- summary(model)$coefficients
        or_table <- data.frame(
            logOR = coefs[, "Estimate"],
            logOR_lower = coefs[, "Estimate"] - 1.96 * coefs[, "Std. Error"],
            logOR_upper = coefs[, "Estimate"] + 1.96 * coefs[, "Std. Error"],
            OR    = exp(coefs[, "Estimate"]),
            lower = exp(coefs[, "Estimate"] - 1.96 * coefs[, "Std. Error"]),
            upper = exp(coefs[, "Estimate"] + 1.96 * coefs[, "Std. Error"])
        )
        print(round(or_table, 3))
    ''')

    # Check whether proportional odds assumption holds
    robjects.r('''
        # Fit separate binary logistic models for each threshold and compare coefficients
        library(brant)
        # brant test is for polr models, so refit with MASS::polr
        library(MASS)
        model_polr <- polr(rank ~ group * criterion, data = dat, Hess = TRUE)
        brant::brant(model_polr)
    ''')

    print(long.groupby(["group", "criterion"])["rank"].value_counts().unstack(fill_value=0))

## Testing effects on AGILE conference papers

The code below takes up after the data preparation section. 

In [ ]:
group_a = pd.concat([group_a1, group_a2], ignore_index=True)
print(len(group_a), group_a.head())

In [ ]:
analyze_group(group_a, df_authors)

For all interpretations, it is important to keep in mind that the data criterion of the pre-intervention group is the reference, and all p-values or odd ratios for the methods and results criteria reference that. The model output shows that the model coefficient for the data criterion of the post-intervention (groupA2) is different from that of the pre-intervention group at the p<0.001 level, and the coefficients for methods and results criteria from the post-intervention group (groupA2:criterionmethods and groupA2:criterionresults) do not differ statistically signficantly from it. 

The pairwise group comparison per criterion offers a more intuitive approach to understanding the results, because it directly addresses our main question: are the ranks of each criterion significantly different between the pre- and post-intervention groups?  

The outputs are clear: The ranks for each criterion are different between groups at p<0.0001.

To determine the direction and size of the effect, we look at the (log)OR:

The OR for criterionmethods (OR = 3.093) and criterionresults (OR = 2.626) show that in the pre-intervention group, methods and results already score higher than data on average. This is a baseline difference between criteria, and not related to the intervention.

For the reference criterion of data, the OR for groupA2 (the post-intervention group) is 19.271 (CI: 10.726–34.624). This means that for the reference criterion (data), post-intervention papers have ~19x higher odds of being in a higher rank category compared to pre-intervention. This is a large, significant improvement for data after intervention. The CI does not include 1, confirming significance.

The effect of the intervention is not meaningfully different for the other two criteria: For groupA2:criterionmethods, the OR is 1.063 (CI: 0.571–1.979), indicating almost the same effect, while for groupA2:criterionresults the OR is 0.640 (CI: 0.344–1.189), indicating a somewhat weaker effect than for data, but still CI crosses 1 so it is not significant.

However, the omnibus test (p≈0) indicates the proportional odds assumption is violated overall. The p=0 and p=1 extremes for criterionresults and groupA2:criterionresults might indicate a numerical issue, possibly a sparse cell (few observations at some rank levels for that criterion), which the last output table confirms. 

We therefore have to interpret the OR carefully, especially specific numeric comparisons. However, the bottom line is that:

1. We can reject H-Null: There has been a statistically significant change in ranks between pre-intervention group and post-intervention group.
2. The OR are large and positive, indicating higher ranks in the post-intervention group.

## Testing effects on GIScience papers

In [3]:
group_b = pd.concat([group_b1, group_b2], ignore_index=True)
print(len(group_b), group_b.head())

NameError: name 'group_b1' is not defined

In [ ]:
analyze_group(group_b, df_authors)

The pairwise group comparison shows that ranks for data are not significantly different (p = 0.0683) between pre- and post-intervention groups, however, those for methods and results criteria are at p < 0.0001. 

The interpretation for the effect size would be that the reference criterion data has negligible or no improvement. However, again methods and results are different from data: the adjusted OR for methods is 1.995 x 4.193 = 8.365035 and for results it is 1.995 x 2.671 = 5.328645, indicating a clearly positive change after the intervention in AGILE.

Like for AGILE, the proportional odds assumption is violated, meaning that we have to interpret actual numbers with care. The bottom line of the analysis is:

1. We can reject H-Null for two criteria: While the ranks for data have not change in a statistically significant way, those for methods and results have.
2. Like for AGILE, the OR are clearly positive for methods and results, so the change is towards higher ranks for the post-intervention group.

## Comparison of changes between conferences

The violation of the proportional odds assumption makes a numerical comparison of the changes between the two conferences difficult. We have considered to use partial proportional odds models or collapsing the ranks into a low (0, 1) and high (2, 3) binary classification. However, neither solves the problem of the limited sample size and distribution of ranks, which results in very few (or none at all) observations for high ranks (2 or 3) in the pre-intervention groups for both conferences. 

We have therefore decided to interpret the outcomes conservatively: We observe overall a notably lower OR for GIScience than for AGILE, and no significant change in the data criterion. Combined with the fact that at least a third of all post-intervention GIScience papers have at least one (co-)author who published also at AGILE, **we are confident to state that based on the statistical testing**, 

1. The improvement of potential reproducibility at AGILE is larger and broader than the improvement at GIScience.
2. The improvement at GIScience might be partially attributable to a spill-over effect through authors who published also at AGILE.
3. Although we cannot prove any causality, the AGILE reproducibility guidelines and the reproducibility review are unlikely to have had a negative effect on the development of paper reproducibility. To the contrary, it seems likely that they have had a positive effect.  